# Demo kickoff / pre-warm

Run this ~5 minutes before presenting. It speeds up the things you can't speed up live:

- Resumes the **warehouse**.
- Resumes and warms the **compute pool** so the promotion ML Job does not pay a cold-start on stage.
- *(optional)* caches the container **image** on the node.
- **Readiness check**: confirms the dev ABT, dev model, and (if promoted) prod predictions exist.

Run the cells top to bottom. Use a session that can assume `ACCOUNTADMIN`.

In [ ]:
from snowflake.snowpark.context import get_active_session
import time

session = get_active_session()

# Object names (mirror config.py, inlined so this notebook is self-contained in a Workspace).
WAREHOUSE         = "CORTEX_CODE_WH"
COMPUTE_POOL      = "MLOPS_CPU_M_POOL"
DEV_DATABASE      = "ML_FRAUD_DEV_SANDBOX"
PROD_DATABASE     = "ML_FRAUD_PRODUCTION"
REGISTRY_SCHEMA   = "ML"
MODEL_NAME        = "AML_FRAUD_GBM"
DEV_ABT           = f"{DEV_DATABASE}.CURATED.FRAUD_ABT"
PREDICTIONS       = f"{PROD_DATABASE}.ANALYTICS.PREDICTIONS"
JOB_PAYLOAD_STAGE = f"{PROD_DATABASE}.{REGISTRY_SCHEMA}.JOB_PAYLOAD"
DEPLOY_ROLE       = "ML_DEPLOY_SVC"

# Set True to also cache the container image (slower, costs more). Default False.
WARM_IMAGE = False

session.sql("USE ROLE ACCOUNTADMIN").collect()
print("session ready as ACCOUNTADMIN")

In [ ]:
# [1] Warehouse - resume so queries and the batch task run immediately.
try:
    session.sql(f"ALTER WAREHOUSE {WAREHOUSE} RESUME").collect()
except Exception:
    pass  # already running
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
session.sql("SELECT 1").collect()
print(f"{WAREHOUSE} ready")

In [ ]:
# [2] Compute pool - resume and wait until a node is ready (no cold-start for the promotion ML Job).
def pool_row():
    for r in session.sql(f"SHOW COMPUTE POOLS LIKE '{COMPUTE_POOL}'").collect():
        return r.as_dict()
    return {}

st = pool_row().get("state")
if st in ("SUSPENDED", "STOPPING"):
    try:
        session.sql(f"ALTER COMPUTE POOL {COMPUTE_POOL} RESUME").collect()
    except Exception as e:
        print("resume:", type(e).__name__, str(e)[:80])

waited = 0
while True:
    r = pool_row()
    st = r.get("state")
    ready = int(r.get("active_nodes", 0) or 0) + int(r.get("idle_nodes", 0) or 0)
    print(f"[{waited}s] state={st} nodes_ready={ready}")
    if st in ("ACTIVE", "IDLE") and ready >= 1:
        break
    if waited > 420:
        print("WARNING: pool not fully warm after 7 min; continuing")
        break
    time.sleep(20); waited += 20
print(f"{COMPUTE_POOL} warm")

In [ ]:
# [2b] OPTIONAL - cache the container image on the node so the real promotion job starts fast.
# Enable by setting WARM_IMAGE = True in the setup cell above.
if WARM_IMAGE:
    import os, tempfile
    from snowflake.ml.jobs import submit_file
    session.sql(f"USE ROLE {DEPLOY_ROLE}").collect()
    session.sql(f"USE SCHEMA {PROD_DATABASE}.{REGISTRY_SCHEMA}").collect()
    with tempfile.TemporaryDirectory() as d:
        p = os.path.join(d, "warm.py")
        open(p, "w").write("print('image warm')\n")
        job = submit_file(p, COMPUTE_POOL, stage_name=JOB_PAYLOAD_STAGE, session=session)
        job.wait()
        print("warm job", job.status)
    session.sql("USE ROLE ACCOUNTADMIN").collect()
else:
    print("WARM_IMAGE=False - skipping image warm")

In [ ]:
# [3] Readiness check - confirm the core dev artifacts exist before presenting.
def count(fqn):
    try:
        return session.sql(f"SELECT COUNT(*) AS C FROM {fqn}").collect()[0]["C"]
    except Exception:
        return None

dev_abt = count(DEV_ABT)
preds   = count(PREDICTIONS)
try:
    from snowflake.ml.registry import Registry
    reg = Registry(session=session, database_name=DEV_DATABASE, schema_name=REGISTRY_SCHEMA)
    dev_versions = [str(v.version_name) for v in reg.get_model(MODEL_NAME).versions()]
except Exception:
    dev_versions = []

print(f"dev ABT rows:       {dev_abt}")
print(f"dev model versions: {dev_versions}")
print(f"prod predictions:   {preds}")

ok = bool(dev_abt) and ("V1" in dev_versions)
print("\n" + ("GO - core dev artifacts present." if ok else
              "CHECK - dev artifacts missing; run setup 00-03 first."))
print("Pool is warm. The promotion ML Job should start without a cold-start.")